In [3]:
from api_calls import read_json
config = read_json('data','config')


FileNotFoundError: [Errno 2] No such file or directory: 'data/config.txt'

In [8]:
import os, math, time, requests, datetime as dt
import pandas as pd
from zoneinfo import ZoneInfo

def get_spreads_and_results_last_week(
    sport: str = "americanfootball_nfl",
    region: str = "us",
    bookmaker_prefs = ("pinnacle","draftkings","fanduel","betmgm","caesars"),
    snapshot_minutes_before_kick: int = 2,
    tz_name: str = "America/New_York",
    api_key: str | None = None,
) -> pd.DataFrame:
    """
    Returns DataFrame:
      ['commence','home_team','home_score','home_point','away_team','away_score','away_point','bookmaker','result','margin']
    result ∈ {'home_cover','away_cover','push', None}
    """
    API = "https://api.the-odds-api.com/v4"
    key_dct = {"key_free":"685f879ea79649c48a2ed648a7876f1f",
               "key_paid":"8f2863c1fffb9d994a5b359fce7802ba"}
    api_key =key_dct['key_paid']
    key = api_key or os.getenv("THE_ODDS_API_KEY")
    if not key:
        raise RuntimeError("Set THE_ODDS_API_KEY or pass api_key")

    tz = ZoneInfo(tz_name)
    today = dt.datetime.now(tz).date()
    last_monday = (today - dt.timedelta(days=today.weekday()+7))
    last_sunday = last_monday + dt.timedelta(days=6)
    start_dt = dt.datetime.combine(last_monday, dt.time(0,0), tzinfo=tz)
    end_dt   = dt.datetime.combine(last_sunday, dt.time(23,59,59), tzinfo=tz)

    def iso_z(dtobj):
        return dtobj.astimezone(dt.timezone.utc).replace(tzinfo=None).isoformat(timespec="seconds")+"Z"

    # 1) scores
    scores_url = f"{API}/sports/{sport}/scores/"
    print(scores_url)
    games = requests.get(scores_url, params={"apiKey": key, "daysFrom": 10}, timeout=20).json()
    rows = []
    print(games)
    for g in games:
        k = dt.datetime.fromisoformat(g["commence_time"].replace("Z","+00:00")).astimezone(tz)
        if not (start_dt <= k <= end_dt):
            continue
        if not g.get("completed"):
            continue
        sc = {s["name"]: int(s["score"]) for s in g.get("scores", []) if s.get("score") is not None}
        rows.append({
            "event_id": g["id"],
            "commence": k,
            "home_team": g["home_team"],
            "away_team": g["away_team"],
            "home_score": sc.get(g["home_team"]),
            "away_score": sc.get(g["away_team"]),
        })
    if not rows:
        return pd.DataFrame(columns=["commence","home_team","home_score","home_point","away_team","away_score","away_point","bookmaker","result","margin"])
    df = pd.DataFrame(rows)

    # 2) historical odds snapshot (≈closing)
    def pick_bm(bookmakers):
        by_key = {bm["key"]: bm for bm in bookmakers}
        for pref in bookmaker_prefs:
            if pref in by_key:
                return by_key[pref]
        return bookmakers[0] if bookmakers else None

    def extract_spreads(ev):
        bm = pick_bm(ev.get("bookmakers", []))
        if not bm:
            return (None, None, None)
        mkt = next((m for m in bm.get("markets", []) if m.get("key")=="spreads"), None)
        if not mkt:
            return (bm["key"], None, None)
        hp = ap = None
        for o in mkt.get("outcomes", []):
            if o.get("name")==ev.get("home_team"): hp = o.get("point")
            elif o.get("name")==ev.get("away_team"): ap = o.get("point")
        return (bm["key"], hp, ap)

    odds_rows = []
    for _, r in df[["event_id","commence"]].iterrows():
        snap = r["commence"] - dt.timedelta(minutes=snapshot_minutes_before_kick)
        url = f"{API}/historical/sports/{sport}/events/{r['event_id']}/odds"
        params = {"apiKey": key, "regions": region, "markets": "spreads", "date": iso_z(snap)}
        try:
            data = requests.get(url, params=params, timeout=20).json()
        except Exception:
            data = {}
        bm, hp, ap = extract_spreads(data) if data else (None, None, None)
        odds_rows.append({"event_id": r["event_id"], "bookmaker": bm, "home_point": hp, "away_point": ap})
        time.sleep(0.2)  # rate friendly
    df = df.merge(pd.DataFrame(odds_rows), on="event_id", how="left")

    # 3) ATS result
    def ats(row):
        hs, as_, hp, ap = row.home_score, row.away_score, row.home_point, row.away_point
        if None in (hs, as_, hp, ap): return pd.Series({"result": None, "margin": None})
        home_net, away_net = hs + hp, as_ + ap
        if math.isclose(home_net, away_net, abs_tol=1e-9): res = "push"
        elif home_net > away_net: res = "home_cover"
        else: res = "away_cover"
        return pd.Series({"result": res, "margin": hs - as_})

    df[["result","margin"]] = df.apply(ats, axis=1)
    out = df[["commence","home_team","home_score","home_point","away_team","away_score","away_point","bookmaker","result","margin"]]
    return out.sort_values("commence").reset_index(drop=True)


In [9]:
df = get_spreads_and_results_last_week()  # NFL, US region
# df has the spreads (≈closing) + scores + who covered
df

https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores/
{'message': 'Invalid daysFrom', 'error_code': 'INVALID_SCORES_DAYS_FROM', 'details_url': 'https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from'}


TypeError: string indices must be integers

In [10]:
import datetime
import json

In [40]:
def getNextWeeksSpreads(dates=None, key_type='free'):
    import requests
    # config = read_json('data','config')
    key_dct = {"key_free":"685f879ea79649c48a2ed648a7876f1f",
               "key_paid":"8f2863c1fffb9d994a5b359fce7802ba"}
    API_KEY = key_dct[f'key_{key_type}']
    SPORT = 'americanfootball_nfl'
    REGIONS = 'us'
    MARKETS = 'h2h,spreads,totals,scores'
    ODDS_FORMAT = 'american'
    # get max date in dates and convert to this format ''2023-10-10T12:15:00Z'

    if key_type=='free':
        DATE = datetime.datetime.today().strftime('%Y-%m-%d')
    else:
        DATE = min(dates)
        datetime_format = '%Y-%m-%dT%H:%M:%SZ'
        minDate = datetime.datetime.strptime(min(dates), datetime_format).date()
        maxDate = datetime.datetime.strptime(max(dates), datetime_format).date()

    if key_type == 'free': url = f'https://api.the-odds-api.com/v4/sports/americanfootball_nfl/odds?regions=us&markets=h2h,spreads,totals&oddsFormat=american&apiKey={API_KEY}'
    else: url = f'https://api.the-odds-api.com/v4/historical/sports/{SPORT}/odds?apiKey={API_KEY}&regions={REGIONS}&markets={MARKETS}&oddsFormat={ODDS_FORMAT}'
    if key_type == 'paid': url = url + f'&date={DATE}'

    r = requests.get(url)
    if r.status_code == 200:
        # for p in r.content:
        #     print(p)
        spreaddata = json.loads(r.content)
    breakpoint()
    print('check spreaddata')
    if key_type == 'paid': spreaddata = spreaddata['data']
    #     hist_spreaddata = json.loads(r.content)
    # spreaddata = hist_spreaddata['data']

    print(f'number of games in spreaddata before date filter = {len(spreaddata)}')
    # filter out games that are not in the date range
    # spreaddata = [game for game in spreaddata if game['commence_time'] >= minDate and game['commence_time'] <= maxDate]
    if key_type == 'paid':
        spreaddata = [game for game in spreaddata if datetime.datetime.strptime(game['commence_time'],datetime_format).date() >= minDate and datetime.datetime.strptime(game['commence_time'],datetime_format).date() <= maxDate]
        print(f'number of games in spreaddata after date filter = {len(spreaddata)}')
    print('Remaining requests', r.headers['x-requests-remaining'])
    print('Used requests', r.headers['x-requests-used'])
    breakpoint()
    return spreaddata
getNextWeeksSpreads(dates=[2024-12-31], key_type='free')


check spreaddata


UnboundLocalError: local variable 'spreaddata' referenced before assignment

In [33]:
def getLastWeeksResults(dates=None, key_type='free'):
    import requests
    # config = read_json('data','config')
    key_dct = {"key_free":"685f879ea79649c48a2ed648a7876f1f",
               "key_paid":"8f2863c1fffb9d994a5b359fce7802ba"}
    API_KEY = key_dct[f'key_{key_type}']
    SPORT = 'americanfootball_nfl'
    REGIONS = 'us'
    MARKETS = 'h2h,spreads,totals,scores'
    ODDS_FORMAT = 'american'
    # get max date in dates and convert to this format ''2023-10-10T12:15:00Z'

    # if key_type=='free':
    DATE = datetime.datetime.today().strftime('%Y-%m-%d')
    # else:
    #     DATE = min(dates)
    datetime_format = '%Y-%m-%dT%H:%M:%SZ'
    # minDate = datetime.datetime.strptime(DATE, datetime_format).date()
    # maxDate = datetime.datetime.strptime(DATE, datetime_format).date()

    if key_type == 'free': url = f'https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores?regions=us&markets=h2h,spreads,totals&oddsFormat=american&apiKey={API_KEY}'
    else: url = f'https://api.the-odds-api.com/v4/historical/sports/{SPORT}/scores?apiKey={API_KEY}&regions={REGIONS}&markets={MARKETS}&oddsFormat={ODDS_FORMAT}'
    if key_type == 'paid': url = url + f'&date={DATE}'

    r = requests.get(url)
    print(url)
    if r.status_code == 200:
        # for p in r.content:
        #     print(p)
        spreaddata = json.loads(r.content)
    breakpoint()
    print('check spreaddata')
    if key_type == 'paid': spreaddata = spreaddata['data']
    #     hist_spreaddata = json.loads(r.content)
    # spreaddata = hist_spreaddata['data']

    print(f'number of games in spreaddata before date filter = {len(spreaddata)}')
    # filter out games that are not in the date range
    # spreaddata = [game for game in spreaddata if game['commence_time'] >= minDate and game['commence_time'] <= maxDate]
    print(spreaddata)
    if key_type == 'paid':
        spreaddata = [game for game in spreaddata if datetime.datetime.strptime(game['commence_time'],datetime_format).date() >= minDate and datetime.datetime.strptime(game['commence_time'],datetime_format).date() <= maxDate]
        print(f'number of games in spreaddata after date filter = {len(spreaddata)}')
    print('Remaining requests', r.headers['x-requests-remaining'])
    print('Used requests', r.headers['x-requests-used'])
    breakpoint()
    return spreaddata
getLastWeeksResults(dates=[2024-12-31], key_type='paid')

https://api.the-odds-api.com/v4/historical/sports/americanfootball_nfl/scores?apiKey=8f2863c1fffb9d994a5b359fce7802ba&regions=us&markets=h2h,spreads,totals,scores&oddsFormat=american&date=2025-09-04
check spreaddata


UnboundLocalError: local variable 'spreaddata' referenced before assignment

In [23]:
def getLastWeeksResults(key_type='free', sport='americanfootball_nfl'):
    import os, json, requests, datetime as dt
    from zoneinfo import ZoneInfo

    # keys (env overrides your hardcoded defaults)
    key_dct = {"key_free":"685f879ea79649c48a2ed648a7876f1f",
               "key_paid":"8f2863c1fffb9d994a5b359fce7802ba"}
    API_KEY = key_dct[f'key_{key_type}']

    API = "https://api.the-odds-api.com/v4"
    tz = ZoneInfo("America/New_York")

    # last Mon..Sun window in local time
    today = dt.datetime.now(tz).date()
    last_monday = today - dt.timedelta(days=today.weekday() + 7)
    last_sunday = last_monday + dt.timedelta(days=6)

    def in_window(commence_iso):
        # API times are Zulu; convert to local date for window check
        k = dt.datetime.fromisoformat(commence_iso.replace("Z", "+00:00")).astimezone(tz).date()
        return last_monday <= k <= last_sunday

    # fetch recent scores (grab enough days, then filter)
    url = f"{API}/sports/{sport}/scores/"
    r = requests.get(url, params={"apiKey": API_KEY, "daysFrom": 14}, timeout=20)
    r.raise_for_status()
    games = json.loads(r.content)

    # keep completed games in last week, return tidy dicts
    results = []
    for g in games:
        if not (g.get("completed") and in_window(g["commence_time"])):
            continue
        sc = {s["name"]: int(s["score"]) for s in g.get("scores", []) if s.get("score") is not None}
        results.append({
            "event_id": g["id"],
            "commence_time": g["commence_time"],
            "home_team": g["home_team"],
            "away_team": g["away_team"],
            "home_score": sc.get(g["home_team"]),
            "away_score": sc.get(g["away_team"]),
        })

    # optional usage headers if present
    if 'x-requests-remaining' in r.headers:
        print('Remaining requests', r.headers['x-requests-remaining'])
        print('Used requests', r.headers.get('x-requests-used'))

    return results
last_week = getLastWeeksResults(key_type='paid')  # NFL
# last_week -> list of dicts with home/away final scores for last week
last_week

HTTPError: 422 Client Error: Unprocessable Entity for url: https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores/?apiKey=8f2863c1fffb9d994a5b359fce7802ba&daysFrom=14

In [41]:
def getLastWeeksResults(dates=None, key_type='free', sport='americanfootball_nfl'):
    import requests, json, datetime
    from zoneinfo import ZoneInfo

    # your keys (same pattern as other fn)
    key_dct = {
        "key_free": "685f879ea79649c48a2ed648a7876f1f",
        "key_paid": "8f2863c1fffb9d994a5b359fce7802ba"
    }
    API_KEY = key_dct[f'key_{key_type}']
    API = "https://api.the-odds-api.com/v4"
    tz = ZoneInfo("America/New_York")
    ISOZ = "%Y-%m-%dT%H:%M:%SZ"

    # --- normalize dates -> local dates ---
    def to_date(x):
        if isinstance(x, datetime.date) and not isinstance(x, datetime.datetime):
            return x
        s = str(x)
        try:
            return datetime.datetime.strptime(s, "%Y-%m-%d").date()
        except ValueError:
            return datetime.datetime.strptime(s, ISOZ).date()

    minDate = maxDate = None
    if dates:
        ds = [to_date(d) for d in dates]
        minDate, maxDate = min(ds), max(ds)

    # --- choose daysFrom window for /scores ---
    # If no dates given: pull recent 7.
    # If dates given: cover minDate..today with buffer, but API may cap (often 3–10 days).
    today_local = datetime.datetime.now(tz).date()
    if minDate:
        days_from = max(1, (today_local - minDate).days + 1)
        # add a 1-day buffer for TZ/late games
        days_from = min(days_from + 1, 10)  # API generally caps this; we'll also back off on 422
    else:
        days_from = 7

    # --- fetch with 422 backoff on daysFrom ---
    url = f"{API}/sports/{sport}/scores/"
    games = None
    for df_try in (days_from, 7, 5, 3, 2, 1):
        r = requests.get(url, params={"apiKey": API_KEY, "daysFrom": df_try}, timeout=20)
        print(r.url)
        if r.status_code == 422:
            continue
        r.raise_for_status()
        games = r.json()
        break
    if games is None:
        raise RuntimeError("Scores endpoint rejected all daysFrom attempts; try a more recent date range.")

    # --- filter & shape ---
    def in_window(commence_iso):
        dloc = datetime.datetime.fromisoformat(commence_iso.replace("Z","+00:00")).astimezone(tz).date()
        return (minDate is None) or (minDate <= dloc <= (maxDate or dloc))

    results = []
    for g in games:
        if not g.get("completed"):
            continue
        if not in_window(g["commence_time"]):
            continue
        sc = {s["name"]: int(s["score"]) for s in g.get("scores", []) if s.get("score") is not None}
        results.append({
            "event_id": g["id"],
            "commence_time": g["commence_time"],
            "home_team": g["home_team"],
            "away_team": g["away_team"],
            "home_score": sc.get(g["home_team"]),
            "away_score": sc.get(g["away_team"]),
        })

    # usage headers if present
    if 'x-requests-remaining' in r.headers:
        print("Remaining requests", r.headers['x-requests-remaining'])
    if 'x-requests-used' in r.headers:
        print("Used requests", r.headers['x-requests-used'])

    # if your requested dates are too far back for scores window, results may be empty
    if minDate and results == []:
        print("Note: The Scores endpoint only returns recent games (provider limit). "
              "For older weeks you must store results yourself or use another results provider.")

    return results


In [47]:
rows = getLastWeeksResults(dates=['2025-8-20','2025-08-27'], key_type='paid')
rows

https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores/?apiKey=8f2863c1fffb9d994a5b359fce7802ba&daysFrom=10
https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores/?apiKey=8f2863c1fffb9d994a5b359fce7802ba&daysFrom=7
https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores/?apiKey=8f2863c1fffb9d994a5b359fce7802ba&daysFrom=5
https://api.the-odds-api.com/v4/sports/americanfootball_nfl/scores/?apiKey=8f2863c1fffb9d994a5b359fce7802ba&daysFrom=3
Remaining requests 19970.0
Used requests 30
Note: The Scores endpoint only returns recent games (provider limit). For older weeks you must store results yourself or use another results provider.


[]

In [48]:
def getLastYearScores(year=None, include_postseason=False, include_preseason=False):
    """
    Returns list of dicts:
      {'event_id','commence_time','week','stage','home_team','away_team','home_score','away_score'}
    Uses your existing API-Sports integration (getGameResults).
    """
    import json
    import datetime as dt
    from zoneinfo import ZoneInfo
    from api_calls import getGameResults

    tz = ZoneInfo("America/New_York")
    if year is None:
        year = dt.datetime.now(tz).year - 1  # last year

    raw = getGameResults(year)  # bytes
    data = json.loads(raw.decode("utf-8"))

    stages = {"Regular Season"}
    if include_postseason: stages.add("Postseason")
    if include_preseason: stages.add("Preseason")

    out = []
    for game in data.get("response", []):
        g = game.get("game", {})
        stage = g.get("stage")
        status_short = g.get("status", {}).get("short")  # 'FT' etc.
        if stage not in stages:
            continue
        if status_short in ("NS", None):  # skip not-started/incomplete
            continue

        date_str = g.get("date", {}).get("date")
        time_str = g.get("date", {}).get("time")
        if not (date_str and time_str):
            continue
        commence_iso = f"{date_str}T{time_str}:00Z"

        teams = game.get("teams", {})
        home = teams.get("home", {}).get("name")
        away = teams.get("away", {}).get("name")
        scores = game.get("scores", {})
        hs = scores.get("home", {}).get("total")
        as_ = scores.get("away", {}).get("total")
        if None in (home, away, hs, as_):
            continue

        out.append({
            "event_id": str(g.get("id") or f"{year}-{home}-{away}-{commence_iso}"),
            "commence_time": commence_iso,
            "week": g.get("week"),
            "stage": stage,
            "home_team": home,
            "away_team": away,
            "home_score": int(hs),
            "away_score": int(as_),
        })
    return out


In [49]:
rows = getLastYearScores()
rows

FileNotFoundError: [Errno 2] No such file or directory: 'data/config.txt'

In [ ]:
# nfl_scores_llm.py
import os, time, json, re
from typing import List, Dict, Any
import requests
import pandas as pd

# ---- OpenAI client (Responses API) ----
from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

ESPN_URL = "https://www.espn.com/nfl/scoreboard/_/week/{week}/year/{year}/seasontype/2"
CBS_URL  = "https://www.cbssports.com/nfl/scoreboard/{year}/regular/{week}/"
PFR_URL  = "https://www.pro-football-reference.com/years/{year}/week_{week}.htm"

def _get(url: str, timeout: int = 25, tries: int = 3) -> str:
    last_err = None
    headers = {
        "User-Agent": "Mozilla/5.0 (scores-bot; +https://chat.openai.com)"
    }
    for _ in range(tries):
        try:
            r = requests.get(url, timeout=timeout, headers=headers)
            if r.status_code == 200 and r.text:
                return r.text
            last_err = RuntimeError(f"HTTP {r.status_code} for {url}")
        except Exception as e:
            last_err = e
        time.sleep(0.8)
    raise last_err

def _html_sources_for_week(year: int, week: int) -> List[Dict[str, Any]]:
    urls = [
        ("espn", ESPN_URL.format(year=year, week=week)),
        ("cbs",  CBS_URL.format(year=year, week=week)),
        ("pfr",  PFR_URL.format(year=year, week=week)),
    ]
    sources = []
    for name, url in urls:
        try:
            html = _get(url)
            sources.append({"name": name, "url": url, "html": html[:500_000]})  # cap size
        except Exception as e:
            # Skip silently; we’ll proceed with what we have
            pass
    if not sources:
        raise RuntimeError("Could not fetch any scoreboard pages.")
    return sources

_JSON_SCHEMA = {
    "name": "nfl_week_scores_schema",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "games": {
                "type": "array",
                "items": {
                    "type": "object",
                    "required": ["date", "home_team", "away_team", "home_score", "away_score", "status"],
                    "properties": {
                        "date":        {"type": "string", "description": "ISO date or source date label"},
                        "home_team":   {"type": "string"},
                        "away_team":   {"type": "string"},
                        "home_score":  {"type": "integer"},
                        "away_score":  {"type": "integer"},
                        "status":      {"type": "string", "description": "e.g., FINAL"},
                        "source_hint": {"type": "string"}
                    }
                }
            },
            "notes": {"type": "string"}
        },
        "required": ["games"]
    }
}

def _llm_extract_scores(year: int, week: int, sources: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Ask the LLM to parse the HTML blobs and return consistent scores for REGULAR SEASON Week 'week' of 'year'.
    """
    sys = (
        "You are a careful data extraction assistant. "
        "Extract NFL REGULAR SEASON scores for the specified week/year from the provided HTML. "
        "Ignore preseason or postseason. If sources disagree, prefer majority; otherwise prefer ESPN, then CBS, then PFR. "
        "Only include games marked final (use 'FINAL' or equivalent). "
        "Team names should be their common names (e.g., 'New York Jets', 'Los Angeles Rams'). "
        "Return only the JSON requested by the schema."
    )

    # Build a compact prompt with multiple sources
    parts = [f"Extract REGULAR SEASON Week {week} of {year}.\n"]
    for s in sources:
        parts.append(f"\n=== SOURCE: {s['name']} | {s['url']} ===\n")
        parts.append(s["html"])
    user = "\n".join(parts)

    resp = client.responses.create(
        model="gpt-5.1",  # or "gpt-4.1" if preferred in your account
        messages=[
            {"role": "system", "content": sys},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_schema", "json_schema": _JSON_SCHEMA},
        temperature=0
    )
    content = resp.output_text  # JSON string
    return json.loads(content)

def _normalize_team(name: str) -> str:
    # Light normalization to handle minor variations
    name = re.sub(r"\s+", " ", name).strip()
    # Expand common abbreviations if any appear (optional: add more mappings as needed)
    fixes = {
        "LA Rams": "Los Angeles Rams",
        "LA Chargers": "Los Angeles Chargers",
        "NY Giants": "New York Giants",
        "NY Jets": "New York Jets",
        "WAS Commanders": "Washington Commanders",
        "TB Buccaneers": "Tampa Bay Buccaneers",
        "NO Saints": "New Orleans Saints",
    }
    return fixes.get(name, name)

def get_nfl_scores_openai(year: int, week: int) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
      ['date','home_team','away_team','home_score','away_score','status']
    Only FINAL games for the requested REGULAR SEASON week.
    """
    sources = _html_sources_for_week(year, week)
    data = _llm_extract_scores(year, week, sources)

    games = data.get("games", [])
    if not games:
        raise RuntimeError("No games parsed by LLM.")
    df = pd.DataFrame(games)
    # Normalize / reorder cols
    for col in ["home_team", "away_team"]:
        if col in df:
            df[col] = df[col].astype(str).map(_normalize_team)
    # Keep only expected columns
    cols = ["date", "home_team", "away_team", "home_score", "away_score", "status"]
    for c in cols:
        if c not in df:
            df[c] = None
    df = df[cols]
    # Drop obvious non-finals if any slipped through
    df = df[df["status"].str.upper().str.contains("FINAL", na=False)].reset_index(drop=True)
    return df

if __name__ == "__main__":
    # Example: Week 18 of the 2024 regular season
    df = get_nfl_scores_openai(2024, 18)
    print(df.to_string(index=False))
